----------- Agent -----------------

1- 创建模型

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
# 加载环境变量
load_dotenv()

model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

print(model)

metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-openai': '1.5.1'}} client=<openai.resources.chat.completions.completions.Completions object at 0x113b0c190> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11473c410> root_client=<openai.OpenAI object at 0x113b30250> root_async_client=<openai.AsyncOpenAI object at 0x11472ff90> model_name='MiniMax-M3' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.minimaxi.com/v1' stream_chunk_timeout=120.0


2-创建agent

In [3]:
from langchain.agents import create_agent

agent= create_agent(
    model=model,
    tools=[]
)
print(agent)

3-工具

In [ ]:
from langchain.tools import tool

@tool
def get_weather()->str:
    """模拟天气查询工具"""
    return "北京天气不错，21度适合长袖"

agent_for_weather = create_agent(
    model=model,
    tools=[get_weather]
)

4-系统提示词

In [6]:
agent_1 = create_agent(
    model=model,
    system_prompt="你是一个优秀的个人助手"
)

5-结构化输出

就是让最终结果返回一个model结构，按照用户定义的模型生成

In [10]:
from pydantic import BaseModel
class Answer(BaseModel):
    cityName: str
    personalCount: str

s_agent = create_agent(
    model=model,
    system_prompt="you are smart",
    response_format=Answer
)

result = s_agent.invoke({"messages":[{"role":"user","content":"北京人口多少人"}]})

print(f"{result}\n -----------")

print(f"{result.keys()}\n -----------")

print(f"{result['structured_response']}\n -----------")

{'messages': [HumanMessage(content='北京人口多少人', additional_kwargs={}, response_metadata={}, id='aef10d29-1e5c-4674-95c2-cee988fc49a6'), AIMessage(content="<think>The user is asking about the population of Beijing. I should note that I don't have real-time data access, but I can provide information based on my knowledge. The tool available seems to be for asking about city population, but it requires specific parameters. Let me use the tool to answer.\n\nActually, looking at the tool, it takes cityName and personalCount as parameters. The personalCount seems to be the answer value. This is a bit unusual - it seems like the tool might be designed for me to provide the answer through it. Let me use it appropriately.\n\nBased on my knowledge (cutoff January 2026), Beijing's permanent population was approximately 21.89 million as of recent census data (2020 census showed about 21.89 million). Let me provide this through the tool.</think>\n\n北京的常住人口约为 **2,189 万人**（约2189万），这是根据2020年第七次全国人口普查的数据

6-Agent state（状态）

记录agent一系列执行内容，历史记录

In [ ]:
from langchain.agents import AgentState

class myAgentState(AgentState):
    user_id: str
    call_count: int

state_agent = create_agent(
    model=model,
    state_schema=myAgentState
)

7-invocation 调用

你可以通过一条消息来调用一个代理。在后台，这条消息会将更新传递给代理的状态。所有代理的状态中都包含一系列消息；要调用代理，请同时传入一条新消息和一个 thread_id，以便代理能够持久化并恢复对话历史：